# Prosody Extraction for 555 Videos (with Checkpointing)

**FIX:** Use ffmpeg to handle ANY audio/video format.
**NEW:** Checkpoint every 25 videos - resume if interrupted!

**Data alignment:**
- WavLM: 555 videos (wavlm_utterance_safe) ✅
- Labels: 555 videos (utterances_clean.jsonl) ✅
- Audio: 555 videos ✅

In [ ]:
# @title Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# @title Step 2: Install ffmpeg + Imports
!apt-get update -qq && apt-get install -qq -y ffmpeg > /dev/null 2>&1
print('ffmpeg installed!')

import os
import json
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import time
import subprocess
import tempfile

In [ ]:
# @title Step 3: Setup paths
BASE_DIR = Path('/content/gdrive/MyDrive')
WAVLM_DIR = BASE_DIR / 'wavlm_utterance_safe'
OUTPUT_FILE = BASE_DIR / 'prosody_555.json'
CHECKPOINT_FILE = BASE_DIR / 'prosody_555_checkpoint.json'

SR = 16000
CHECKPOINT_EVERY = 25  # Save checkpoint every N videos

AUDIO_FOLDERS = [
    'chuckle_audio',
    'chuckle_audio_all/audio',
    'chuckle_audio_all/audio_new',
    'chuckle_audio_all/audio_final',
    'chuckle_audio_all/audio_all'
]

In [ ]:
# @title Step 4: Load or init checkpoint
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            return json.load(f)
    return None

def save_checkpoint(prosody_data, processed_vids, failed_vids):
    checkpoint = {
        'prosody_data': prosody_data,
        'processed_videos': list(processed_vids),
        'failed_videos': list(failed_vids),
        'timestamp': time.time()
    }
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f)
    print(f'Checkpoint saved! ({len(processed_vids)} videos done)')

# Try to load existing checkpoint
checkpoint = load_checkpoint()
if checkpoint:
    print(f'Found checkpoint! {len(checkpoint["processed_videos"])} videos already done.')
    START_FROM = set(checkpoint['processed_videos'])
    prosody_data = checkpoint['prosody_data']
    failed_videos = set(checkpoint['failed_videos'])
else:
    print('No checkpoint found. Starting fresh.')
    START_FROM = set()
    prosody_data = {}
    failed_videos = set()

In [ ]:
# @title Step 5: Download utterances
possible_utt_paths = [
    BASE_DIR / 'utterances_clean.jsonl',
    Path('/content/utterances_clean.jsonl'),
    Path('/content/gdrive/utterances_clean.jsonl'),
]

utt_path = None
for p in possible_utt_paths:
    if p.exists():
        utt_path = p
        print(f'Found: {p}')
        break

if utt_path is None:
    print('Downloading...')
    subprocess.run(['rclone', 'copy', 'gdrive:utterances_clean.jsonl', '/content/'], check=True)
    utt_path = Path('/content/utterances_clean.jsonl')

print(f'Using: {utt_path}')

In [ ]:
# @title Step 6: Load WavLM video IDs
wavlm_files = list(WAVLM_DIR.glob('*.json'))
wavlm_files = [f for f in wavlm_files if f.name != 'checkpoint.json']

wavlm_video_ids = set()
for f in wavlm_files:
    with open(f) as fh:
        data = json.load(fh)
        wavlm_video_ids.add(data['video_id'])

remaining = wavlm_video_ids - START_FROM - failed_videos
print(f'WavLM videos: {len(wavlm_video_ids)}')
print(f'Already done: {len(START_FROM)}')
print(f'Remaining: {len(remaining)}')

In [ ]:
# @title Step 7: Build audio path lookup
audio_paths = {}

for folder in AUDIO_FOLDERS:
    folder_path = BASE_DIR / folder
    if not folder_path.exists():
        print(f'  {folder}: NOT FOUND')
        continue

    files = list(folder_path.iterdir())
    count = 0
    for f in files:
        if f.name.endswith('.part'):
            continue
        if f.suffix in ['.wav', '.mp3', '.m4a', '.mp4', '.webm', '.flac', '.aac']:
            vid = f.stem
            if vid not in audio_paths:
                audio_paths[vid] = str(f)
                count += 1
    print(f'  {folder}: {count} files')

print(f'\nTotal: {len(audio_paths)}')

In [ ]:
# @title Step 8: Load utterances
from collections import defaultdict

utterances_by_video = defaultdict(list)

with open(utt_path) as f:
    for line in f:
        d = json.loads(line)
        vid = d['video_id']
        if vid in wavlm_video_ids:
            utterances_by_video[vid].append({
                'start': d['start'],
                'end': d['end'],
                'label': d.get('label', 0),
            })

print(f'Videos with utterances: {len(utterances_by_video)}')

In [ ]:
# @title Step 9: Audio loading with ffmpeg
def load_audio_ffmpeg(audio_path, target_sr=16000):
    with tempfile.NamedTemporaryFile(suffix='.raw', delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cmd = ['ffmpeg', '-y', '-i', audio_path, '-f', 's16le', '-acodec', 'pcm_s16le', '-ar', str(target_sr), '-ac', '1', tmp_path]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
        if result.returncode != 0:
            y, sr = librosa.load(audio_path, sr=target_sr, mono=True)
            return y.astype(np.float32), sr
        with open(tmp_path, 'rb') as f:
            raw_data = f.read()
        y = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
        return y, target_sr
    finally:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

In [ ]:
# @title Step 10: Prosody extraction function
def extract_prosody_21dim(y, sr):
    features = []
    
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    try:
        rms = librosa.feature.rms(y=y)[0]
        features.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms) - np.min(rms)])
    except:
        features.extend([0]*5)
    
    try:
        duration = len(y) / sr
        speech_rate = np.sum(rms > np.mean(rms)) / duration if duration > 0 else 0
        features.extend([duration, speech_rate])
    except:
        features.extend([0]*2)
    
    try:
        spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
        spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
        spec_flat = librosa.feature.spectral_flatness(y=y)[0]
        zcr = librosa.feature.zero_crossing_rate(y)[0]
        features.extend([np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat), np.mean(zcr), np.std(zcr)])
    except:
        features.extend([0]*5)
    
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
        features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    except:
        features.extend([0]*4)
    
    return np.array(features, dtype=np.float32)

In [ ]:
# @title Step 11: Extract prosody with checkpointing
print(f'Starting prosody extraction...')
print(f'Already done: {len(START_FROM)} videos')
print(f'Remaining: {len(remaining)} videos')
t0 = time.time()
total_utts = 0

for i, vid in enumerate(tqdm(remaining, desc='Videos')):
    # Check if already done or failed
    if vid in START_FROM or vid in failed_videos:
        continue
    
    audio_path = audio_paths.get(vid)
    if not audio_path:
        failed_videos.add(vid)
        continue
    
    try:
        y, sr = load_audio_ffmpeg(audio_path, SR)
    except Exception as e:
        print(f'\nError loading {vid}: {e}')
        failed_videos.add(vid)
        continue
    
    utts = utterances_by_video.get(vid, [])
    if not utts:
        failed_videos.add(vid)
        continue
    
    video_prosody = []
    for utt in utts:
        start_s = utt['start']
        end_s = utt['end']
        
        start_sample = int(start_s * SR)
        end_sample = int(end_s * SR)
        
        if end_sample > len(y):
            end_sample = len(y)
        if start_sample >= len(y):
            start_sample = 0
        
        y_slice = y[start_sample:end_sample]
        
        if len(y_slice) < SR * 0.1:
            video_prosody.append({'start': start_s, 'end': end_s, 'prosody': np.zeros(21).tolist(), 'label': utt['label']})
        else:
            prosody = extract_prosody_21dim(y_slice, SR)
            video_prosody.append({'start': start_s, 'end': end_s, 'prosody': prosody.tolist(), 'label': utt['label']})
        
        total_utts += 1
    
    prosody_data[vid] = video_prosody
    
    # Checkpoint every N videos
    if (len(START_FROM) + len(prosody_data)) % CHECKPOINT_EVERY == 0:
        save_checkpoint(prosody_data, START_FROM | set(prosody_data.keys()), failed_videos)

elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(f'Videos: {len(prosody_data)}/{len(wavlm_video_ids)}')
print(f'Failed: {len(failed_videos)}')
print(f'Utterances: {total_utts}')

In [ ]:
# @title Step 12: Save final output
# Save checkpoint one more time
save_checkpoint(prosody_data, START_FROM | set(prosody_data.keys()), failed_videos)

print(f'\nSaving to {OUTPUT_FILE}...')

output = {
    'video_count': len(prosody_data),
    'total_utterances': sum(len(v) for v in prosody_data.values()),
    'prosody': prosody_data
}

with open(OUTPUT_FILE, 'w') as f:
    json.dump(output, f)

print(f'Saved! Size: {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

In [ ]:
# @title Step 13: Verify
print('=== VERIFICATION ===')

with open(OUTPUT_FILE) as f:
    data = json.load(f)

print(f"Videos: {data['video_count']}")
print(f"Utterances: {data['total_utterances']}")

pos_count = sum(utt['label'] for utts in data['prosody'].values() for utt in utts)
total = data['total_utterances']
print(f"Positive: {pos_count}/{total} ({pos_count/total*100:.1f}%)")

print('\n✅ Extraction complete!')